# Notebook 06 — Evaluation & Robustesse

**Projet :** Prediction du risque d'abandon scolaire  
**Equipe :** Hugo RAGUIN · Amine TALEB · Elliot FIORESE  

Ce notebook evalue **rigoureusement** les modeles entraines en Notebook 05 :

1. Choix et justification des metriques
2. Tableau comparatif complet
3. Validation croisee StratifiedKFold (5 plis)
4. Courbes ROC comparatives
5. Matrices de confusion
6. Analyse de l'importance des variables
7. Analyse de sensibilite au seuil de decision
8. Conclusion et recommandation

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath('..')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score
)

from utils_viz import set_custom_style

set_custom_style(theme='light')
%matplotlib inline
RANDOM_STATE = 42
print('Environnement pret.')

In [ ]:
# Chargement des donnees et reentrainement des modeles
df = pd.read_csv('data/processed/tp1_student_risk_model_ready.csv')
y = df['dropout_risk'].astype(int)
X = df.drop(columns=['dropout_risk'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

baseline = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)

logistic = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))
])
logistic.fit(X_train, y_train)
logistic_prob = logistic.predict_proba(X_test)[:, 1]
logistic_pred = logistic.predict(X_test)

rf = RandomForestClassifier(
    n_estimators=400, class_weight='balanced_subsample',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:, 1]
rf_pred = rf.predict(X_test)

print('Modeles entraines : Baseline, Logistic, Random Forest')

## 1. Choix et justification des metriques

Dans un systeme d'alerte precoce, **le cout des erreurs n'est pas symetrique** :

| Erreur | Consequence | Cout relatif |
|---|---|---|
| **Faux negatif** (etudiant a risque non detecte) | L'etudiant abandonne sans intervention | **Eleve** |
| **Faux positif** (intervention inutile) | Ressource pedagogique mobilisee inutilement | Faible |

=> **Metrique prioritaire : rappel**. Secondairement : F1-score et ROC-AUC.

In [ ]:
# Tableau comparatif complet
def full_eval(name, y_true, y_pred, y_prob=None):
    r = {
        'Modele': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Rappel': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_prob) if y_prob is not None else float('nan'),
        'AP': average_precision_score(y_true, y_prob) if y_prob is not None else float('nan')
    }
    return r

rows = [
    full_eval('Baseline', y_test, baseline.predict(X_test)),
    full_eval('Logistic Regression', y_test, logistic_pred, logistic_prob),
    full_eval('Random Forest', y_test, rf_pred, rf_prob),
]
metrics_df = pd.DataFrame(rows).set_index('Modele')
print('=== Tableau comparatif ===')
print(metrics_df.round(3).to_string())

## 2. Validation croisee StratifiedKFold (5 plis)

La validation croisee stratifiee evalue la **robustesse et la generalisation** des modeles. Elle permet de detecter le sur-apprentissage et de verifier la stabilite des scores.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'recall', 'f1', 'roc_auc']

cv_lr = cross_validate(logistic, X, y, cv=cv, scoring=scoring)
cv_rf = cross_validate(rf, X, y, cv=cv, scoring=scoring)

cv_df = pd.DataFrame({
    'Metrique': ['Accuracy', 'Rappel', 'F1', 'ROC-AUC'],
    'LR mean': [cv_lr[f'test_{m}'].mean() for m in ['accuracy', 'recall', 'f1', 'roc_auc']],
    'LR std':  [cv_lr[f'test_{m}'].std()  for m in ['accuracy', 'recall', 'f1', 'roc_auc']],
    'RF mean': [cv_rf[f'test_{m}'].mean() for m in ['accuracy', 'recall', 'f1', 'roc_auc']],
    'RF std':  [cv_rf[f'test_{m}'].std()  for m in ['accuracy', 'recall', 'f1', 'roc_auc']],
}).set_index('Metrique')

print('=== Validation croisee StratifiedKFold (5 plis) ===')
print(cv_df.round(3).to_string())

In [ ]:
# Boxplots : stabilite des scores par pli
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric, title in [
    (axes[0], 'test_recall', 'Rappel (5 plis CV)'),
    (axes[1], 'test_roc_auc', 'ROC-AUC (5 plis CV)')
]:
    bp = ax.boxplot([cv_lr[metric], cv_rf[metric]], patch_artist=True, widths=0.4)
    for patch, color in zip(bp['boxes'], ['#1A73E8', '#D93025']):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticklabels(['Logistic', 'Random Forest'])
    ax.set_title(title)
    ax.set_ylabel('Score')

fig.suptitle('Stabilite des modeles — Validation croisee 5 plis stratifies', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()

## 3. Courbes ROC et Precision-Rappel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Courbes ROC ---
ax = axes[0]
for prob, label, color in [(logistic_prob, 'Logistic', '#1A73E8'), (rf_prob, 'Random Forest', '#D93025')]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{label} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'--', color='#70757A', linewidth=1, label='Aleatoire (AUC=0.500)')
ax.set_xlabel('Taux de faux positifs')
ax.set_ylabel('Taux de vrais positifs')
ax.set_title('Courbes ROC')
ax.legend(loc='lower right')

# --- Courbes Precision-Rappel ---
ax = axes[1]
for prob, label, color in [(logistic_prob, 'Logistic', '#1A73E8'), (rf_prob, 'Random Forest', '#D93025')]:
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    ax.plot(rec, prec, color=color, linewidth=2, label=f'{label} (AP={ap:.3f})')
ax.axhline(y=y_test.mean(), color='#70757A', linestyle='--', linewidth=1, label='Baseline (prevalence)')
ax.set_xlabel('Rappel')
ax.set_ylabel('Precision')
ax.set_title('Courbes Precision-Rappel')
ax.legend(loc='upper right')

fig.suptitle('Comparaison des performances — ROC et Precision-Rappel', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()

## 4. Matrices de confusion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, pred, title in [
    (axes[0], logistic_pred, 'Logistic Regression'),
    (axes[1], rf_pred, 'Random Forest')
]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non a risque', 'A risque'],
                yticklabels=['Non a risque', 'A risque'])
    ax.set_xlabel('Predit')
    ax.set_ylabel('Reel')
    ax.set_title(title)
    tn, fp, fn, tp_ = cm.ravel()
    ax.set_xlabel(f'Predit  (VP={tp_}, FP={fp}, FN={fn}, VN={tn})')

fig.suptitle('Matrices de confusion — Comparaison des modeles', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()

## 5. Analyse de sensibilite au seuil de decision

Par defaut, le seuil est fixe a 0.5. En le faisant varier, on peut ajuster le **compromis precision / rappel** selon les besoins metier.

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)
recalls, precisions, f1s = [], [], []

for t in thresholds:
    pred_t = (logistic_prob >= t).astype(int)
    recalls.append(recall_score(y_test, pred_t, zero_division=0))
    precisions.append(precision_score(y_test, pred_t, zero_division=0))
    f1s.append(f1_score(y_test, pred_t, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, recalls, color='#D93025', linewidth=2, label='Rappel')
ax.plot(thresholds, precisions, color='#188038', linewidth=2, label='Precision')
ax.plot(thresholds, f1s, color='#1A73E8', linewidth=2, linestyle='--', label='F1-score')
ax.axvline(0.5, color='black', linewidth=1, linestyle=':', label='Seuil par defaut (0.5)')
best_t = thresholds[np.argmax(f1s)]
ax.axvline(best_t, color='#F29900', linewidth=1, linestyle=':', label=f'Seuil optimal F1 ({best_t:.2f})')
ax.set_xlabel('Seuil de decision')
ax.set_ylabel('Score')
ax.set_title('Sensibilite au seuil de decision — Logistic Regression')
ax.legend()
fig.tight_layout()
plt.show()

print(f'Seuil optimal (max F1) : {best_t:.2f}  |  F1 max : {max(f1s):.3f}')

## 6. Importance des variables — analyse detaillee

In [ ]:
fi = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
fi = fi.sort_values('importance', ascending=False)

# Importance cumulee (combien de variables pour 80 % de l'explication ?)
fi['cumulative'] = fi['importance'].cumsum()
n80 = (fi['cumulative'] <= 0.80).sum() + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 12 features
top12 = fi.head(12)
axes[0].barh(top12['feature'][::-1], top12['importance'][::-1], color='#1A73E8', edgecolor='none')
axes[0].set_xlabel('Importance (Gini)')
axes[0].set_title('Top 12 variables explicatives')

# Importance cumulee
axes[1].plot(range(1, len(fi)+1), fi['cumulative'], color='#1A73E8', linewidth=2)
axes[1].axhline(0.80, color='#D93025', linestyle='--', linewidth=1, label='Seuil 80 %')
axes[1].axvline(n80, color='#F29900', linestyle=':', linewidth=1, label=f'{n80} variables')
axes[1].set_xlabel('Nombre de variables')
axes[1].set_ylabel('Importance cumulee')
axes[1].set_title('Importance cumulee des variables')
axes[1].legend()

fig.suptitle('Analyse de l\'importance des variables — Random Forest', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()

print(f'{n80} variables expliquent 80 % de la variance — potentiel de reduction du modele.')

## 7. Rapport de classification detaille et conclusion

In [ ]:
print('=== Rapport complet — Logistic Regression (modele retenu) ===')
print(classification_report(y_test, logistic_pred, target_names=['Non a risque', 'A risque']))

print('=== Rapport complet — Random Forest ===')
print(classification_report(y_test, rf_pred, target_names=['Non a risque', 'A risque']))

## 8. Conclusion et recommandation

| Critere | Logistic Regression | Random Forest | Gagnant |
|---|---|---|---|
| **Rappel** (prioritaire) | ~0.74 | ~0.30 | **Logistic** |
| **ROC-AUC** | ~0.88 | ~0.84 | **Logistic** |
| **Accuracy** | ~0.83 | ~0.93 | Random Forest |
| **Interpretabilite** | Coefficients lineaires | Boite noire | **Logistic** |
| **Stabilite CV** | std faible | std faible | Ex-aequo |

**Modele retenu : Regression Logistique**

Elle offre le meilleur rappel (detecce ~74 % des etudiants a risque) et reste interpretable par les equipes pedagogiques. Le Random Forest est utile pour le classement des variables mais son rappel trop faible (~30 %) le disqualifie pour un usage de prevention.

In [ ]:
# Sauvegarde des predictions et metriques finales
pred_df = X_test.copy()
pred_df['y_true'] = y_test.values
pred_df['y_pred_lr'] = logistic_pred
pred_df['y_prob_lr'] = logistic_prob
pred_df['y_pred_rf'] = rf_pred
pred_df['y_prob_rf'] = rf_prob
pred_df.head(200).to_csv('data/processed/tp3_predictions_sample.csv', index=False)
print('Fichiers de resultats sauvegardes dans data/processed/')